In [1]:
import torch

torch.__version__

'2.5.1+cu124'

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

In [3]:
torch.manual_seed(4242)

In [4]:
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        # "../data/p1ch2/mnist",
        "./data/p1ch2/mnist",
        train=True,
        download=True,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=64,
    shuffle=True,
)

In [5]:
!ls -al ./data/p1ch2/mnist

total 12
drwxr-xr-x 3 root root 4096 Feb 21 07:54 .
drwxr-xr-x 3 root root 4096 Feb 21 07:54 ..
drwxr-xr-x 3 root root 4096 Feb 21 07:54 MNIST


In [6]:
torch.cuda.is_available()

True

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [8]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [9]:
model = Net()
model.to(device)  # Model.to 是一个 in-place 方法
model

Net(
  (conv1): Conv2d(1, 10, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(10, 20, kernel_size=(5, 5), stride=(1, 1))
  (conv2_drop): Dropout2d(p=0.5, inplace=False)
  (fc1): Linear(in_features=320, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=10, bias=True)
)

In [10]:
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

In [11]:
for epoch in range(10):
    for batch_idx, (data, target) in enumerate(train_loader):
        data = data.to(device)  # Tensor.to 不是一个 in-place 方法
        target = target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
    print("Current loss", float(loss))

Current loss 0.5038483142852783
Current loss 0.3051275312900543
Current loss 0.311644583940506
Current loss 0.1089656800031662
Current loss 0.4026910364627838
Current loss 0.14981238543987274
Current loss 0.05207217484712601
Current loss 0.10083332657814026
Current loss 0.1372600793838501
Current loss 0.11393605917692184


In [12]:
# torch.save(model.state_dict(), "../data/p1ch2/mnist/mnist.pth")
torch.save(model.state_dict(), "./data/p1ch2/mnist/mnist.pth")

In [13]:
pretrained_model = Net()
# pretrained_model.load_state_dict(torch.load("../data/p1ch2/mnist/mnist.pth"))
pretrained_model.load_state_dict(torch.load("./data/p1ch2/mnist/mnist.pth"))

<ipython-input-13-448887552d20>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_model.load_state_dict(torch.load("./data/p1ch2/mnist/mnist.pth"))


<All keys matched successfully>